# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [149]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [150]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [151]:
print ('Shape:', df.shape)
print ('\nData Types:')
print (df.dtypes)
print ('\nNull count per column')
print (df.isna().sum())
print ('\nDuplicate rows:', df.duplicated().sum())

Shape: (8, 6)

Data Types:
order_id      int64
item         object
category     object
qty         float64
price        object
ts           object
dtype: object

Null count per column
order_id    0
item        1
category    0
qty         1
price       0
ts          1
dtype: int64

Duplicate rows: 1


**What is wrong with this data?** List at least five specific problems:

1. price is an object
2. null item
3. null qty
4. null ts
5. duplicated row

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [152]:
removed = df.duplicated().sum()   # TODO: how many duplicates were there?
clean = df.drop_duplicates().copy()  # TODO: df with duplicates dropped, copied

log('Step 1: duplicates', 'dropped exact duplicate rows', removed)

[Step 1: duplicates] dropped exact duplicate rows (1 row(s))


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [153]:
clean['price'] = pd.to_numeric(clean['price'].str.strip().str.replace("$",'',regex=False))

assert clean['price'].dtype == float
log('Step 2: price into a real number', 'stripped the dollar sign and any whitespace, then converted price to a float\n', clean)

[Step 2: price into a real number] stripped the dollar sign and any whitespace, then converted price to a float
 (   order_id           item   category  qty  price                   ts
0         1   Cheeseburger       Food  2.0    7.5  2026-09-05T12:03:00
2         2  cheese burger       food  1.0    7.5     09/05/2026 12:40
3         3    Foam Finger      Merch  NaN   12.0  2026-09-05 13:00:00
4         4   UVA T-Shirt     Apparel  2.0   24.0     2026-09-05 13:05
5         5    Rain Poncho   RainGear -3.0    6.0  2026-09-05T13:20:00
6         6    rain poncho  rain-gear  4.0    6.0                  NaN
7         7            NaN      Merch  1.0   12.0  2026-09-05T14:00:00 row(s))


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [154]:
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

missing = clean['qty'].isna().sum()    # TODO: count of NaN quantities
negative = (clean['qty'] < 0).sum()   # TODO: count of negative quantities

clean = clean.dropna(subset=['qty']).copy()
log('Step 3: no quantity', 'Dropped rows with missing quantity', missing)

clean = clean[clean['qty'] >= 0].copy()
log('Step 3: refund', 'Dropped rows with a negative quantity', negative)
# TODO: apply your decision, then log both separately

[Step 3: no quantity] Dropped rows with missing quantity (1 row(s))
[Step 3: refund] Dropped rows with a negative quantity (1 row(s))


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [155]:
print('before:', sorted(clean['category'].unique()))
started = clean['category'].nunique()

clean['category'] = clean['category'].str.lower().str.strip().str.replace('-','',regex=True)   # TODO: lowercase, strip, remove punctuation

CATEGORY_MAP = {'merch':'apparel', 'merch':'merch', 'raingear':'raingear', 'food':'food'}   # TODO: CATEGORY_MAP = {...} for the judgment calls
clean['category'] = clean['category'].replace(CATEGORY_MAP)

ended = clean['category'].nunique()
print('after: ', sorted(clean['category'].unique()))

log('Step 4: categories that mean one thing', 'normalized case/punctuation and mapped category variants', ('Started:', started, 'Ended:', ended))

before: ['Apparel', 'Food', 'Merch', 'food', 'rain-gear']
after:  ['apparel', 'food', 'merch', 'raingear']
[Step 4: categories that mean one thing] normalized case/punctuation and mapped category variants (('Started:', 5, 'Ended:', 4) row(s))


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [156]:
print('before:', sorted(clean['item'].dropna().unique()))

clean['item'] = (clean['item'].str.lower().str.strip().str.replace(' ', '', regex=False))

missing_item = clean['item'].isna().sum()

clean = clean.dropna(subset=['item']).copy()

print('after:', clean['item'])

log('Step 5: item names', 'normalized item names and dropped rows with missing item', missing_item)

before: ['Cheeseburger', 'UVA T-Shirt ', 'cheese burger', 'rain poncho']
after: 0    cheeseburger
2    cheeseburger
4      uvat-shirt
6      rainponcho
Name: item, dtype: object
[Step 5: item names] normalized item names and dropped rows with missing item (1 row(s))


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [157]:
clean['ts'] = pd.to_datetime(clean['ts'], errors='coerce')

failed = clean['ts'].isna().sum()

print('Failed timestamps:', failed)

clean['hour'] = clean['ts'].dt.hour

log('Step 6: timestamps','parsed timestamps and created hour column',failed)

Failed timestamps: 3
[Step 6: timestamps] parsed timestamps and created hour column (3 row(s))


### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [158]:
# TODO: assertions
assert clean['qty'].notna().all()
assert (clean['qty'] >= 0).all()
assert clean['price'].dtype == float
assert clean['category'].notna().all()
assert clean['item'].notna().all()

clean['revenue'] = clean['qty'] * clean['price']      # TODO: clean['revenue'] = ...

# TODO: print rows, units, revenue, distinct categories
print('Revenue by row:')
print(clean[['order_id', 'item', 'qty', 'price', 'revenue']])
print('\nTotal revenue:', clean['revenue'].sum())

Revenue by row:
   order_id          item  qty  price  revenue
0         1  cheeseburger  2.0    7.5     15.0
2         2  cheeseburger  1.0    7.5      7.5
4         4    uvat-shirt  2.0   24.0     48.0
6         6    rainponcho  4.0    6.0     24.0

Total revenue: 94.5


### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [159]:
show_log()

,step,decision,rows
0,Step 1: duplicates,dropped exact duplicate rows,1
1,Step 2: price into a real number,"stripped the dollar sign and any whitespace, t...",order_id item category qty pr...
2,Step 3: no quantity,Dropped rows with missing quantity,1
3,Step 3: refund,Dropped rows with a negative quantity,1
4,Step 4: categories that mean one thing,normalized case/punctuation and mapped categor...,"(Started:, 5, Ended:, 4)"
5,Step 5: item names,normalized item names and dropped rows with mi...,1
6,Step 6: timestamps,parsed timestamps and created hour column,3


**The decision that mattered most:** Deciding what to do with the refund mattered most, because it changed the true total revenue, whereas dropping duplicates just created a more accurate report.

**Revenue with refund:** $76.50  **Revenue without refund:** $94.50

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [164]:
# Checkpoint
rows_after = len(clean)
revenue_after = clean['revenue'].sum
biggest_decision = 'Deciding what to do with the refund mattered most, because it changed the true total revenue'
revenue_other_way = '$76.50'

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)

rows after cleaning: 4
revenue: <bound method Series.sum of 0    15.0
2     7.5
4    48.0
6    24.0
Name: revenue, dtype: float64>
decision that mattered: Deciding what to do with the refund mattered most, because it changed the true total revenue
revenue the other way: $76.50
